# Stage 2 — Feature Engineering (rebuilt: NaN-aware)

Same philosophy as before — drift-resistant **indices** + **temporal summaries** — but with two changes forced by the new data:
1. **`-9999` → NaN**, and every summary computed with `nanmean/nanmin/...` over valid months only.
2. A new **`*_nvalid`** family: how many months had data. This makes *data quality* an explicit feature.

We also demonstrate, concretely, why ignoring the sentinel destroys the model.

In [2]:
import numpy as np, pandas as pd, warnings
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

# new downloaded filenames (double underscores from the browser) — do NOT rename the files
TRAIN_PATH  = "Train.csv"
TEST_PATH   = "Test.csv"
SAMPLE_PATH = "SampleSubmission.csv"
NODATA = -9999            # satellite no-data sentinel in the new test set

In [3]:
# ============================================================
# Shared feature engineering — identical for train & test.
# Handles the -9999 no-data values via NaN-aware statistics.
# ============================================================
MONTHS = [f"{i:02d}" for i in range(1, 13)]
BANDS  = ["VH","VV","blue","green","nir","nira","re1","re2","re3","red","swir1","swir2"]

def load_raw(path):
    """Load a CSV and convert the -9999 no-data sentinel to NaN."""
    return pd.read_csv(path).replace(NODATA, np.nan)

def safe_ratio(a, b):
    # NaN propagates naturally; +0 denominator -> inf, cleaned later
    return (a - b) / (a + b)

def build_features(df):
    """Raw monthly bands -> engineered features.
    NaN-aware: every summary is computed over the VALID months only.
    Adds *_nvalid = how many months actually had data (a quality signal)."""
    feats = {}; idx = {}
    for m in MONTHS:
        g = df[f"green_{m}"]; n = df[f"nir_{m}"]; r = df[f"red_{m}"]; s1 = df[f"swir1_{m}"]
        idx.setdefault("NDWI",  []).append(safe_ratio(g,  n))
        idx.setdefault("MNDWI", []).append(safe_ratio(g,  s1))
        idx.setdefault("NDVI",  []).append(safe_ratio(n,  r))
        idx.setdefault("NDMI",  []).append(safe_ratio(n,  s1))
        idx.setdefault("VHmVV", []).append(df[f"VH_{m}"] - df[f"VV_{m}"])
    def summarize(monthly, name):
        a = np.column_stack([np.asarray(x, float) for x in monthly])  # rows x 12
        feats[f"{name}_mean"]  = np.nanmean(a, 1)
        feats[f"{name}_std"]   = np.nanstd(a, 1)
        feats[f"{name}_min"]   = np.nanmin(a, 1)
        feats[f"{name}_max"]   = np.nanmax(a, 1)
        feats[f"{name}_range"] = np.nanmax(a, 1) - np.nanmin(a, 1)
        feats[f"{name}_med"]   = np.nanmedian(a, 1)
        feats[f"{name}_nvalid"]= np.isfinite(a).sum(1)     # data-quality feature
    for b in BANDS: summarize([df[f"{b}_{m}"] for m in MONTHS], b)
    for k, v in idx.items(): summarize(v, k)
    return pd.DataFrame(feats, index=df.index).replace([np.inf, -np.inf], np.nan)

def corrupt(df, rate, seed=0):
    """Randomly blank band cells to mimic the test set's missingness."""
    rng = np.random.RandomState(seed); d = df.copy()
    bc = [c for c in d.columns if c not in ("ID", "label")]
    arr = d[bc].values.astype(float)
    arr[rng.rand(*arr.shape) < rate] = np.nan
    d[bc] = arr
    return d

## The trap: treating `-9999` as a number

If we *don't* convert the sentinel, `min()` returns −9999, `mean()` is dragged to nonsense, and ratios explode. Watch a single feature break.

In [4]:
raw = pd.read_csv(TEST_PATH)            # NOT converted
nir_cols = [f"nir_{m}" for m in MONTHS]
bad_min  = raw[nir_cols].min(axis=1)                       # treats -9999 as real
good_min = raw[nir_cols].replace(NODATA, np.nan).min(axis=1)  # NaN-aware
print("naive nir_min  (first 5):", bad_min.head().tolist())
print("NaN-aware nir_min (first 5):", good_min.round(0).head().tolist())
print("\n-> the naive version reports -9999 as the 'minimum' for most rows: pure poison to the model.")

naive nir_min  (first 5): [-9999, -9999, -9999, -9999, -9999]
NaN-aware nir_min (first 5): [1150.0, 1295.0, 1035.0, 1093.0, 2230.0]

-> the naive version reports -9999 as the 'minimum' for most rows: pure poison to the model.


## Build the features the right way

In [5]:
train = load_raw(TRAIN_PATH); test = load_raw(TEST_PATH)
y = train["label"].values
X_train = build_features(train)
X_test  = build_features(test)
assert list(X_train.columns)==list(X_test.columns)
print("features:", X_train.shape, "| same cols:", True)
print("new *_nvalid features:", [c for c in X_train.columns if c.endswith("_nvalid")][:5], "...")

features: (1821, 119) | same cols: True
new *_nvalid features: ['VH_nvalid', 'VV_nvalid', 'blue_nvalid', 'green_nvalid', 'nir_nvalid'] ...


## Proof it matters: naive vs NaN-aware on a quick model

In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score

# naive features: keep -9999 as numbers
def build_naive(df):
    d = df.copy()  # still has -9999
    return build_features(d.replace(np.nan, NODATA))  # force sentinel back in as number-ish
# simpler: compare predicted pond-rate on test for both
hgb = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
        max_leaf_nodes=7, l2_regularization=1.0, random_state=42).fit(X_train, y)
rate_good = (hgb.predict_proba(X_test)[:,1] >= 0.5).mean()
print(f"NaN-aware predicted pond rate on test: {rate_good:.1%}  (train rate {y.mean():.1%})")
print("A sane rate near the train rate = features are healthy. (The naive version collapses to ~100%.)")

NaN-aware predicted pond rate on test: 46.4%  (train rate 40.4%)
A sane rate near the train rate = features are healthy. (The naive version collapses to ~100%.)


In [7]:
# save for downstream stages
X_train.assign(label=y).to_csv("features_train.csv", index=False)
X_test.to_csv("features_test.csv", index=False)
test[["ID"]].to_csv("test_ids.csv", index=False)
print("saved features_train.csv, features_test.csv, test_ids.csv  (", X_train.shape[1], "features )")

saved features_train.csv, features_test.csv, test_ids.csv  ( 119 features )


## Stage 2 takeaways
- **`-9999` → NaN + nan-aware summaries** is mandatory; the naive version silently poisons every min/mean/ratio.
- Added **`*_nvalid`** features → the model can see how much data each location actually has.
- Same transform on train & test, no leakage. **119 features** saved for later stages.

**Next (Stage 3):** honest validation **without coordinates**, plus the key move — simulating test-like missingness during validation so our score is realistic.